# Profile Cre Mdr Evaluability



Profile CRE and MDR evaluability

This file evaluates whether the ARMD source contains enough
organism and susceptibility information to apply the accepted
phenotypic CRE and MDR Pseudomonas aeruginosa rules.

No phenotype classifications are written to production models
at this stage.

1. Audit source organism labels eligible for CRE evaluation.

Purpose:
Identify every source label that would be assigned to a
CRE-eligible organism group before formalizing the mapping
in a version-controlled seed.

This CASE statement is temporary profiling logic. It will not
become the final production mapping without review.

In [1]:
%sql
with source_organisms as (

    select
        cohort_id,
        organism_source,

        count(*) as culture_organism_rows,
        count(distinct patient_key) as patients

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    group by
        cohort_id,
        organism_source

),

classified as (

    select
        cohort_id,
        organism_source,
        culture_organism_rows,
        patients,

        case

            -- Current and phenotype-qualified E. coli labels.
            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            -- Historical and current Klebsiella aerogenes labels.
            -- This condition must come before the broad
            -- Enterobacter and Klebsiella conditions.
            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            -- Other Klebsiella source labels.
            when organism_source like 'KLEBSIELLA %'
                then 'Klebsiella spp.'

            -- Enterobacter source labels other than historical
            -- Enterobacter aerogenes.
            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

            -- Citrobacter source labels, including historical
            -- labels beginning with ZZZ.
            when organism_source like 'CITROBACTER %'
              or organism_source like 'ZZZCITROBACTER %'
                then 'Citrobacter spp.'

            -- Only Serratia marcescens is included under the
            -- accepted CRE phenotype scope.
            when organism_source like 'SERRATIA MARCESCENS%'
                then 'Serratia marcescens'

            -- Proteus species, including historical labels.
            when organism_source like 'PROTEUS %'
              or organism_source like 'ZZZPROTEUS %'
                then 'Proteus spp.'

            -- Morganella scope is limited to M. morganii.
            when organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

        end as cre_organism_group

    from source_organisms

)

select
    cohort_id,
    cre_organism_group,
    organism_source,
    culture_organism_rows,
    patients

from classified

where cre_organism_group is not null

order by
    cohort_id,
    cre_organism_group,
    culture_organism_rows desc,
    organism_source;

,cohort_id,cre_organism_group,organism_source,culture_organism_rows,patients
0,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX,59,56
1,blood,Citrobacter spp.,CITROBACTER KOSERI,40,39
2,blood,Citrobacter spp.,CITROBACTER FREUNDII,19,18
3,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),9,9
4,blood,Citrobacter spp.,CITROBACTER KOSERI (CARBAPENEM RESISTANT),6,6
5,blood,Citrobacter spp.,CITROBACTER SPECIES,2,2
6,blood,Citrobacter spp.,CITROBACTER YOUNGAE,2,2
7,blood,Citrobacter spp.,ZZZCITROBACTER AMALONATICUS,2,2
8,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,254,240
9,blood,Enterobacter spp.,ENTEROBACTER CLOACAE,63,62


## Reviewed mapping coverage for CRE-eligible source labels

The preceding query uses broad source-label patterns to discover the
organism vocabulary that may be eligible for CRE evaluation.

Those broad patterns are appropriate for profiling, but the production
phenotype model should use an explicit, version-controlled mapping from
each source organism label to its reviewed base organism identity.

This check identifies CRE-eligible source labels that are not yet
included in the reviewed organism mapping.

For each CRE rule group, it also identifies:

- whether imipenem must be excluded from the phenotype rule;
- which unmapped labels already contain an explicit source-system
  carbapenem-resistant designation.

No production mapping is changed by this query.

In [1]:
%sql
with source_labels as (

    select distinct
        organism_source,
        mapping_status

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

classified as (

    select
        organism_source,
        mapping_status,

        case

            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            when organism_source like 'KLEBSIELLA %'
                then 'Klebsiella spp.'

            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

            when organism_source like 'CITROBACTER %'
              or organism_source like 'ZZZCITROBACTER %'
                then 'Citrobacter spp.'

            when organism_source like 'SERRATIA MARCESCENS%'
                then 'Serratia marcescens'

            when organism_source like 'PROTEUS %'
              or organism_source like 'ZZZPROTEUS %'
                then 'Proteus spp.'

            when organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

        end as cre_rule_group

    from source_labels

),

unmapped as (

    select
        organism_source,
        cre_rule_group,

        case
            when organism_source like '%(CARBAPENEM RESISTANT)%'
                then 1
            else 0
        end as source_designated_cre

    from classified

    where cre_rule_group is not null
      and mapping_status = 'unmapped'

)

select
    cre_rule_group,

    case
        when cre_rule_group in (
            'Proteus spp.',
            'Morganella morganii'
        )
            then 1
        else 0
    end as exclude_imipenem_from_cre_rule,

    count(*) as unmapped_source_label_count,

    array_sort(
        collect_set(organism_source)
    ) as unmapped_source_labels,

    array_sort(
        collect_set(
            case
                when source_designated_cre = 1
                    then organism_source
            end
        )
    ) as source_designated_cre_labels

from unmapped

group by cre_rule_group

order by cre_rule_group

,cre_rule_group,exclude_imipenem_from_cre_rule,unmapped_source_label_count,unmapped_source_labels,source_designated_cre_labels
0,Citrobacter spp.,0,11,"[CITROBACTER AMALONATICUS COMPLEX, CITROBACTER BRAAKII, CITROBACTER FREUNDII, CITROBACTER FREUNDII COMPLEX, CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT), CITROBACTER KOSERI, CITROBACTER KOSERI (CARBAPENEM RESISTANT), CITROBACTER SPECIES, CITROBACTER YOUNGAE, ZZZCITROBACTER AMALONATICUS, ZZZCITROBACTER FARMERI]","[CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT), CITROBACTER KOSERI (CARBAPENEM RESISTANT)]"
1,Enterobacter spp.,0,7,"[ENTEROBACTER AMNIGENUS 1, ENTEROBACTER ASBURIAE, ENTEROBACTER CANCEROGENUS, ENTEROBACTER CLOACAE, ENTEROBACTER GERGOVIAE, ENTEROBACTER SPECIES, ENTEROBACTER TAYLORAE]",[]
2,Klebsiella aerogenes,0,3,"[KLEBSIELLA AEROGENES, KLEBSIELLA AEROGENES (CARBAPENEM RESISTANT), ZZZENTEROBACTER AEROGENES]",[KLEBSIELLA AEROGENES (CARBAPENEM RESISTANT)]
3,Klebsiella spp.,0,3,"[KLEBSIELLA OZAENAE, KLEBSIELLA PNEUMONIAE SSP. OZAENAE, KLEBSIELLA SPECIES]",[]
4,Morganella morganii,1,1,[MORGANELLA MORGANII],[]
5,Proteus spp.,1,4,"[PROTEUS SPECIES, PROTEUS VULGARIS GROUP, ZZZPROTEUS PENNERI, ZZZPROTEUS VULGARIS]",[]


## Reporting-group audit for Enterobacter and Citrobacter

The CRE phenotype definition and the antibiogram reporting groups serve
different purposes.

`cre_rule_group` identifies the broad organism group used to determine
whether a culture-organism record is eligible for CRE evaluation.

`organism_reporting_group` defines how reviewed organism identities are
grouped for patient-level deduplication and display in the antibiogram.

This query compares the exact ARMD source vocabulary with those reviewed
SOP reporting rules before any production mapping is changed.

In [2]:
%sql
with vocabulary as (

    select
        organism_source,

        max(organism_standardized)
            as current_standardized_name,

        max(source_qualifier)
            as current_source_qualifier,

        max(mapping_status)
            as current_mapping_status,

        count(*) as culture_organism_rows,

        count(distinct patient_key)
            as patients,

        array_sort(
            collect_set(cohort_id)
        ) as cohorts

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_source like 'ENTEROBACTER %'
       or organism_source like 'ZZZENTEROBACTER %'
       or organism_source like 'KLEBSIELLA AEROGENES%'
       or organism_source like 'CITROBACTER %'
       or organism_source like 'ZZZCITROBACTER %'

    group by organism_source

),

reviewed as (

    select
        *,

        case

            -- Current and historical Klebsiella aerogenes
            -- terminology. This remains separate from the
            -- Enterobacter cloacae complex.
            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'ENTEROBACTER AEROGENES%'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            -- Laboratory SOP reporting group:
            -- Enterobacter cloacae complex.
            when organism_source like 'ENTEROBACTER CLOACAE%'
              or organism_source like 'ENTEROBACTER ASBURIAE%'
              or organism_source like 'ENTEROBACTER HORMAECHEI%'
              or organism_source like 'ENTEROBACTER KOBEI%'
              or organism_source like 'ENTEROBACTER LUDWIGII%'
                then 'Enterobacter cloacae complex'

            -- Laboratory SOP reporting group:
            -- Citrobacter freundii complex.
            when organism_source like 'CITROBACTER BRAAKII%'
              or organism_source like 'CITROBACTER FARMERI%'
              or organism_source like 'ZZZCITROBACTER FARMERI%'
              or organism_source like 'CITROBACTER FREUNDII%'
              or organism_source like 'CITROBACTER KOSERI%'
              or organism_source like 'CITROBACTER YOUNGAE%'
                then 'Citrobacter freundii complex'

        end as proposed_sop_reporting_group,

        case
            when organism_source like '%(CARBAPENEM RESISTANT)%'
                then 1
            else 0
        end as source_designated_cre

    from vocabulary

)

select
    organism_source,
    current_standardized_name,
    current_source_qualifier,
    current_mapping_status,

    proposed_sop_reporting_group,
    source_designated_cre,

    culture_organism_rows,
    patients,
    cohorts

from reviewed

order by
    case
        when proposed_sop_reporting_group =
             'Enterobacter cloacae complex'
            then 1

        when proposed_sop_reporting_group =
             'Klebsiella aerogenes'
            then 2

        when proposed_sop_reporting_group =
             'Citrobacter freundii complex'
            then 3

        else 4
    end,

    organism_source

,organism_source,current_standardized_name,current_source_qualifier,current_mapping_status,proposed_sop_reporting_group,source_designated_cre,culture_organism_rows,patients,cohorts
0,ENTEROBACTER ASBURIAE,None,None,unmapped,Enterobacter cloacae complex,0,7,7,[urine]
1,ENTEROBACTER CLOACAE,None,None,unmapped,Enterobacter cloacae complex,0,380,346,"[blood, respiratory, urine]"
2,ENTEROBACTER CLOACAE COMPLEX,Enterobacter cloacae complex,None,reviewed,Enterobacter cloacae complex,0,1507,1285,"[blood, respiratory, urine]"
3,ENTEROBACTER CLOACAE COMPLEX (CARBAPENEM RESISTANT),Enterobacter cloacae complex,carbapenem_resistant,reviewed,Enterobacter cloacae complex,1,24,22,"[blood, respiratory, urine]"
4,KLEBSIELLA AEROGENES,None,None,unmapped,Klebsiella aerogenes,0,690,568,"[blood, respiratory, urine]"
5,KLEBSIELLA AEROGENES (CARBAPENEM RESISTANT),None,None,unmapped,Klebsiella aerogenes,1,8,7,"[blood, respiratory, urine]"
6,ZZZENTEROBACTER AEROGENES,None,None,unmapped,Klebsiella aerogenes,0,725,639,"[blood, respiratory, urine]"
7,CITROBACTER BRAAKII,None,None,unmapped,Citrobacter freundii complex,0,31,30,"[respiratory, urine]"
8,CITROBACTER FREUNDII,None,None,unmapped,Citrobacter freundii complex,0,202,181,"[blood, respiratory, urine]"
9,CITROBACTER FREUNDII COMPLEX,None,None,unmapped,Citrobacter freundii complex,0,806,701,"[blood, respiratory, urine]"


## CRE source-label and AST concordance

Source organism labels containing `CARBAPENEM RESISTANT` are treated as
source-system phenotype designations, not as definitive classifications.

This analysis independently derives the expanded CRE phenotype from
resolved susceptibility endpoints and compares that result with the
source designation.

The independent AST classification is:

- `positive`: at least one qualifying carbapenem has a resolved
  `Resistant` interpretation;
- `not_detected`: at least one qualifying carbapenem has a resolved
  `Susceptible` or `Intermediate` interpretation, no qualifying
  carbapenem is Resistant, and no qualifying endpoint is missing,
  conflicting, Inconclusive, or Synergism;
- `not_evaluable`: no qualifying Susceptible, Intermediate, or Resistant
  endpoint is available, or a missing, conflicting, Inconclusive, or
  Synergism endpoint prevents a confident `not_detected` classification.

A positive classification takes precedence because one qualifying
Resistant result is sufficient under the accepted phenotype rule.

For Proteus species and Morganella morganii, imipenem and
imipenem/relebactam are not used in the CRE determination.

This audit is performed before patient-level deduplication because the
phenotype category must be assigned before selecting the first isolate
within each reporting category.

In [10]:
%sql
with organism_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,

        case

            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            when organism_source like 'KLEBSIELLA %'
                then 'Klebsiella spp.'

            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

            when organism_source like 'CITROBACTER %'
              or organism_source like 'ZZZCITROBACTER %'
                then 'Citrobacter spp.'

            when organism_source like 'SERRATIA MARCESCENS%'
                then 'Serratia marcescens'

            when organism_source like 'PROTEUS %'
              or organism_source like 'ZZZPROTEUS %'
                then 'Proteus spp.'

            when organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

        end as cre_rule_group,

        case
            when upper(organism_source) like '%CARBAPENEM RESISTANT%'
                then 1
            else 0
        end as source_designated_cre

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

cre_eligible as (

    select *
    from organism_scope
    where cre_rule_group is not null

),

qualifying_results as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,
        e.organism_source,
        e.cre_rule_group,
        e.source_designated_cre,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from cre_eligible e

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on e.culture_organism_key = r.culture_organism_key

       and (

            (
                e.cre_rule_group in (
                    'Proteus spp.',
                    'Morganella morganii'
                )

                and r.antibiotic in (
                    'Meropenem',
                    'Ertapenem',
                    'Meropenem/Vaborbactam'
                )
            )

            or

            (
                e.cre_rule_group not in (
                    'Proteus spp.',
                    'Morganella morganii'
                )

                and r.antibiotic in (
                    'Imipenem',
                    'Meropenem',
                    'Ertapenem',
                    'Meropenem/Vaborbactam',
                    'Imipenem/Relebactam'
                )
            )

       )

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        cre_rule_group,
        source_designated_cre,

        sum(
            case
                when antibiotic is not null then 1
                else 0
            end
        ) as qualifying_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility = 'Resistant'
                    then 1
                else 0
            end
        ) as resistant_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as non_resistant_endpoint_count,

        sum(
            case
                when resolution_status in (
                    'missing',
                    'conflict'
                )
                    then 1

                when resolution_status = 'resolved'
                 and (
                     susceptibility is null
                     or susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count,

        array_sort(
            collect_set(
                case
                    when antibiotic is not null
                        then concat(
                            antibiotic,
                            '=',
                            coalesce(
                                susceptibility,
                                upper(resolution_status)
                            )
                        )
                end
            )
        ) as qualifying_carbapenem_results

    from qualifying_results

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        cre_rule_group,
        source_designated_cre

),

ast_classified as (

    select
        *,

        case
            when resistant_endpoint_count > 0
                then 'positive'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when non_resistant_endpoint_count > 0
                then 'not_detected'

            else 'not_evaluable'
        end as ast_cre_status

    from isolate_summary

),

concordance as (

    select
        *,

        case

            when source_designated_cre = 1
             and ast_cre_status = 'positive'
                then 'concordant_positive'

            when source_designated_cre = 1
             and ast_cre_status = 'not_detected'
                then 'source_positive_ast_not_detected'

            when source_designated_cre = 1
             and ast_cre_status = 'not_evaluable'
                then 'source_positive_not_evaluable'

            when source_designated_cre = 0
             and ast_cre_status = 'positive'
                then 'ast_positive_without_source_label'

            when source_designated_cre = 0
             and ast_cre_status = 'not_detected'
                then 'no_source_label_ast_not_detected'

            else 'no_source_label_not_evaluable'

        end as cre_concordance_status

    from ast_classified

)

select
    cohort_id,
    cre_rule_group,
    cre_concordance_status,

    count(*) as culture_organism_records,

    sum(source_designated_cre)
        as source_designated_cre_records,

    sum(
        case
            when ast_cre_status = 'positive' then 1
            else 0
        end
    ) as ast_cre_positive_records

from concordance

group by
    cohort_id,
    cre_rule_group,
    cre_concordance_status

order by
    cohort_id,
    cre_rule_group,
    cre_concordance_status

,cohort_id,cre_rule_group,cre_concordance_status,culture_organism_records,source_designated_cre_records,ast_cre_positive_records
0,blood,Citrobacter spp.,no_source_label_ast_not_detected,123,0,0
1,blood,Citrobacter spp.,no_source_label_not_evaluable,1,0,0
2,blood,Citrobacter spp.,source_positive_ast_not_detected,15,15,0
3,blood,Enterobacter spp.,ast_positive_without_source_label,8,0,8
4,blood,Enterobacter spp.,concordant_positive,3,3,3
5,blood,Enterobacter spp.,no_source_label_ast_not_detected,320,0,0
6,blood,Escherichia coli,ast_positive_without_source_label,1,0,1
7,blood,Escherichia coli,concordant_positive,4,4,4
8,blood,Escherichia coli,no_source_label_ast_not_detected,3334,0,0
9,blood,Escherichia coli,no_source_label_not_evaluable,1,0,0


## Detailed CRE discordance patterns

The aggregate concordance results show that source phenotype labels and
available categorical AST results do not always agree.

This section summarizes the exact qualifying carbapenem-result patterns
for records in one of three discordant states:

- source-designated CRE without a qualifying Resistant AST result;
- source-designated CRE that cannot be evaluated from available AST;
- AST-derived CRE without a resistant source label.

Results are grouped by organism source label and carbapenem pattern so
that the disagreement can be reviewed without publishing patient- or
culture-level identifiers.

These results will determine whether AST-derived status, source
designation, or an explicitly documented combination should control the
final phenotype field.

In [11]:
%sql
with organism_scope as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,

        case

            when organism_source like 'ESCHERICHIA COLI%'
                then 'Escherichia coli'

            when organism_source = 'ZZZENTEROBACTER AEROGENES'
              or organism_source like 'KLEBSIELLA AEROGENES%'
                then 'Klebsiella aerogenes'

            when organism_source like 'KLEBSIELLA %'
                then 'Klebsiella spp.'

            when organism_source like 'ENTEROBACTER %'
                then 'Enterobacter spp.'

            when organism_source like 'CITROBACTER %'
              or organism_source like 'ZZZCITROBACTER %'
                then 'Citrobacter spp.'

            when organism_source like 'SERRATIA MARCESCENS%'
                then 'Serratia marcescens'

            when organism_source like 'PROTEUS %'
              or organism_source like 'ZZZPROTEUS %'
                then 'Proteus spp.'

            when organism_source = 'MORGANELLA MORGANII'
                then 'Morganella morganii'

        end as cre_rule_group,

        case
            when upper(organism_source) like '%CARBAPENEM RESISTANT%'
                then 1
            else 0
        end as source_designated_cre

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

cre_eligible as (

    select *
    from organism_scope
    where cre_rule_group is not null

),

qualifying_results as (

    select
        e.culture_organism_key,
        e.culture_key,
        e.cohort_id,
        e.patient_key,
        e.organism_source,
        e.cre_rule_group,
        e.source_designated_cre,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from cre_eligible e

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on e.culture_organism_key = r.culture_organism_key

       and (

            (
                e.cre_rule_group in (
                    'Proteus spp.',
                    'Morganella morganii'
                )

                and r.antibiotic in (
                    'Meropenem',
                    'Ertapenem',
                    'Meropenem/Vaborbactam'
                )
            )

            or

            (
                e.cre_rule_group not in (
                    'Proteus spp.',
                    'Morganella morganii'
                )

                and r.antibiotic in (
                    'Imipenem',
                    'Meropenem',
                    'Ertapenem',
                    'Meropenem/Vaborbactam',
                    'Imipenem/Relebactam'
                )
            )

       )

),

isolate_summary as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        cre_rule_group,
        source_designated_cre,

        sum(
            case
                when antibiotic is not null then 1
                else 0
            end
        ) as qualifying_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate',
                     'Resistant'
                 )
                    then 1
                else 0
            end
        ) as evaluable_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility = 'Resistant'
                    then 1
                else 0
            end
        ) as resistant_endpoint_count,

        sum(
            case
                when resolution_status = 'resolved'
                 and susceptibility in (
                     'Susceptible',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as non_resistant_endpoint_count,

        sum(
            case
                when resolution_status in (
                    'missing',
                    'conflict'
                )
                    then 1

                when resolution_status = 'resolved'
                 and (
                     susceptibility is null
                     or susceptibility not in (
                         'Susceptible',
                         'Intermediate',
                         'Resistant'
                     )
                 )
                    then 1

                else 0
            end
        ) as non_evaluable_endpoint_count,

        array_sort(
            collect_set(
                case
                    when antibiotic is not null
                        then concat(
                            antibiotic,
                            '=',
                            coalesce(
                                susceptibility,
                                upper(resolution_status)
                            )
                        )
                end
            )
        ) as qualifying_carbapenem_results

    from qualifying_results

    group by
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,
        cre_rule_group,
        source_designated_cre

),

ast_classified as (

    select
        *,

        case
            when resistant_endpoint_count > 0
                then 'positive'

            when non_evaluable_endpoint_count > 0
                then 'not_evaluable'

            when non_resistant_endpoint_count > 0
                then 'not_detected'

            else 'not_evaluable'
        end as ast_cre_status

    from isolate_summary

),

concordance as (

    select
        *,

        case

            when source_designated_cre = 1
             and ast_cre_status = 'positive'
                then 'concordant_positive'

            when source_designated_cre = 1
             and ast_cre_status = 'not_detected'
                then 'source_positive_ast_not_detected'

            when source_designated_cre = 1
             and ast_cre_status = 'not_evaluable'
                then 'source_positive_not_evaluable'

            when source_designated_cre = 0
             and ast_cre_status = 'positive'
                then 'ast_positive_without_source_label'

            when source_designated_cre = 0
             and ast_cre_status = 'not_detected'
                then 'no_source_label_ast_not_detected'

            else 'no_source_label_not_evaluable'

        end as cre_concordance_status

    from ast_classified

)

select
    cohort_id,
    cre_rule_group,
    organism_source,

    ast_cre_status,
    cre_concordance_status,

    qualifying_endpoint_count,
    evaluable_endpoint_count,
    non_resistant_endpoint_count,
    non_evaluable_endpoint_count,
    resistant_endpoint_count,

    concat_ws(
        ', ',
        qualifying_carbapenem_results
    ) as qualifying_carbapenem_result_pattern,

    count(*) as culture_organism_records

from concordance

where cre_concordance_status in (
    'source_positive_ast_not_detected',
    'source_positive_not_evaluable',
    'ast_positive_without_source_label'
)

group by
    cohort_id,
    cre_rule_group,
    organism_source,
    ast_cre_status,
    cre_concordance_status,
    qualifying_endpoint_count,
    evaluable_endpoint_count,
    non_resistant_endpoint_count,
    non_evaluable_endpoint_count,
    resistant_endpoint_count,
    qualifying_carbapenem_results

order by
    cohort_id,
    culture_organism_records desc,
    cre_rule_group,
    organism_source,
    qualifying_carbapenem_result_pattern

,cohort_id,cre_rule_group,organism_source,ast_cre_status,cre_concordance_status,qualifying_endpoint_count,evaluable_endpoint_count,non_resistant_endpoint_count,non_evaluable_endpoint_count,resistant_endpoint_count,qualifying_carbapenem_result_pattern,culture_organism_records
0,blood,Citrobacter spp.,CITROBACTER KOSERI (CARBAPENEM RESISTANT),not_detected,source_positive_ast_not_detected,2,2,2,0,0,"Imipenem=Susceptible, Meropenem=Susceptible",6
1,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,positive,ast_positive_without_source_label,3,3,2,0,1,"Ertapenem=Resistant, Imipenem=Susceptible, Meropenem=Susceptible",6
2,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),not_detected,source_positive_ast_not_detected,2,2,2,0,0,"Imipenem=Susceptible, Meropenem=Susceptible",5
3,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),not_detected,source_positive_ast_not_detected,3,3,3,0,0,"Ertapenem=Susceptible, Imipenem=Susceptible, Meropenem=Susceptible",2
4,blood,Citrobacter spp.,CITROBACTER FREUNDII COMPLEX (CARBAPENEM RESISTANT),not_detected,source_positive_ast_not_detected,2,2,2,0,0,"Ertapenem=Susceptible, Meropenem=Susceptible",2
5,blood,Enterobacter spp.,ENTEROBACTER CLOACAE COMPLEX,positive,ast_positive_without_source_label,2,2,1,0,1,"Ertapenem=Resistant, Meropenem=Susceptible",2
6,blood,Escherichia coli,ESCHERICHIA COLI,positive,ast_positive_without_source_label,3,3,2,0,1,"Ertapenem=Resistant, Imipenem=Susceptible, Meropenem=Susceptible",1
7,blood,Klebsiella aerogenes,KLEBSIELLA AEROGENES,positive,ast_positive_without_source_label,2,2,1,0,1,"Ertapenem=Resistant, Meropenem=Susceptible",1
8,blood,Klebsiella spp.,KLEBSIELLA PNEUMONIAE,positive,ast_positive_without_source_label,3,3,1,0,2,"Ertapenem=Resistant, Meropenem/Vaborbactam=Susceptible, Meropenem=Resistant",1
9,blood,Klebsiella spp.,KLEBSIELLA PNEUMONIAE,positive,ast_positive_without_source_label,2,2,0,0,2,"Ertapenem=Resistant, Meropenem=Resistant",1


## S. aureus oxacillin-resistance endpoint coverage

Source organism labels containing `MRSA` or `OXACILLIN RESISTANT` are
treated as source-system phenotype designations rather than definitive
resistance classifications.

Before deriving an independent oxacillin-resistance phenotype, this
section profiles the availability and interpretation of the defining
oxacillin and cefoxitin endpoints.

The analysis identifies:

- which defining antibiotics are present in ARMD;
- susceptibility categories and reconciliation states by source label;
- culture-organism records with no defining endpoint;
- whether source-designated resistant records have usable AST evidence.

No production phenotype is assigned by this query.

In [12]:
%sql
with staphylococcus_aureus as (

    select
        culture_organism_key,
        culture_key,
        cohort_id,
        patient_key,
        organism_source,

        case
            when upper(organism_source) like '%MRSA%'
              or upper(organism_source) like '%OXACILLIN RESISTANT%'
                then 1
            else 0
        end as source_designated_oxacillin_resistant

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where organism_standardized = 'Staphylococcus aureus'

),

defining_results as (

    select
        s.culture_organism_key,
        s.culture_key,
        s.cohort_id,
        s.patient_key,
        s.organism_source,
        s.source_designated_oxacillin_resistant,

        r.antibiotic,
        r.resolution_status,
        r.susceptibility

    from staphylococcus_aureus s

    left join workspace.micro_dev.int_susceptibility_reconciliation r
        on s.culture_organism_key = r.culture_organism_key

       and r.antibiotic in (
           'Oxacillin',
           'Cefoxitin'
       )

)

select
    cohort_id,
    organism_source,
    source_designated_oxacillin_resistant,

    coalesce(
        antibiotic,
        '[no defining endpoint]'
    ) as defining_antibiotic,

    coalesce(
        resolution_status,
        'not_present'
    ) as resolution_status,

    coalesce(
        susceptibility,
        '[null]'
    ) as susceptibility,

    count(
        distinct culture_organism_key
    ) as culture_organism_records,

    count(
        distinct patient_key
    ) as patients

from defining_results

group by
    cohort_id,
    organism_source,
    source_designated_oxacillin_resistant,
    coalesce(
        antibiotic,
        '[no defining endpoint]'
    ),
    coalesce(
        resolution_status,
        'not_present'
    ),
    coalesce(
        susceptibility,
        '[null]'
    )

order by
    cohort_id,
    source_designated_oxacillin_resistant desc,
    organism_source,
    defining_antibiotic,
    resolution_status,
    susceptibility

,cohort_id,organism_source,source_designated_oxacillin_resistant,defining_antibiotic,resolution_status,susceptibility,culture_organism_records,patients
0,blood,STAPH AUREUS {MRSA},1,Oxacillin,conflict,[null],3,3
1,blood,STAPH AUREUS {MRSA},1,Oxacillin,resolved,Resistant,418,366
2,blood,STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE),0,Oxacillin,resolved,Resistant,3,3
3,blood,STAPH AUREUS(COLONY VARIANT - SMALL COLONY OR OTHER MORPHOTYPE),0,Oxacillin,resolved,Susceptible,4,4
4,blood,STAPHYLOCOCCUS AUREUS,0,Oxacillin,conflict,[null],1,1
5,blood,STAPHYLOCOCCUS AUREUS,0,Oxacillin,missing,[null],1,1
6,blood,STAPHYLOCOCCUS AUREUS,0,Oxacillin,resolved,Resistant,200,187
7,blood,STAPHYLOCOCCUS AUREUS,0,Oxacillin,resolved,Susceptible,1297,1204
8,blood,STAPHYLOCOCCUS AUREUS,0,[no defining endpoint],not_present,[null],13,13
9,respiratory,STAPH AUREUS {MRSA},1,Oxacillin,conflict,[null],17,13
